# Chunking, Embeddings, and RAG with LlamaIndex + Ollama (Local Only)

# =================================================================

- Take a **long-ish text** and **chunk** it into smaller pieces.
- Use an **embedding model** to represent those chunks as vectors.
- Build a simple **RAG (Retrieval-Augmented Generation)** pipeline:
  - Retrieve the most relevant chunks using embeddings.
  - Let a local LLM (via **Ollama**) generate an answer based on those chunks.

Everything runs **locally** using:

- **Ollama** for the LLM and (optionally) the embedding model.
- **LlamaIndex** for chunking, indexing, and querying.

By the end, you should be able to:

- Explain what **chunking** is and why it helps with context windows.
- Explain what an **embedding** is and how it’s used for retrieval.
- Understand how RAG adds **external knowledge** to an LLM.

Before you run this notebook, ensure the following pre-requisites have met:
Ollama installed (on each machine)
1) Download from: https://ollama.com
2) Double-click and invoke Ollama and ask a question to force it to download the required packages
3) After install, in a terminal or command  window, run:

    ollama pull llama3
    ollama pull nomic-embed-text

Then install Conda environment and Python packages:

    conda create -n rag_ollama python=3.11 -y
    conda activate rag_ollama

    pip install \
      llama-index \
      llama-index-llms-ollama \
      llama-index-embeddings-ollama

In [ ]:
%pip install llama-index-core llama-index-llms-ollama llama-index-embeddings-ollama

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Cell 2: Define a sample "document" and a user question

# This is our "long" document. In real scenarios, this might be
# several pages of a PDF or a long article.
doc_text = """
Imagine you are the head chef at a massive gala in **Boston**, and your job is to prepare appetizers for the guests.

### The Tale of Two Chefs

**Chef Constant ()**
Chef Constant has a signature move: no matter how many guests arrive, he always puts a single cherry on top of a pre-made cake. Whether there is 1 guest or 1,000, it takes him the exact same amount of time. He is remarkably efficient because his work doesn't scale with the size of the crowd.

**Chef Linear ()**
Chef Linear is assigned to peel carrots. If 10 guests arrive, he peels 10 carrots. If 1,000 guests arrive, he peels 1,000 carrots. His workload grows "linearly" with the number of guests (). He’s doing fine for now, but he’s starting to sweat as the guest list grows.

**Chef Quadratic ()**
Chef Quadratic is in charge of "Social Networking." He has to introduce every guest to every other guest.

* If there are 2 guests, he makes 1 introduction.
* If there are 10 guests, he makes nearly 100.
* If 1,000 guests show up, he has to facilitate **1,000,000** introductions.

While the other chefs are finished and cleaning up, Chef Quadratic is still working hours after the gala has ended.

---

### Why Big O Matters

In your **Algorithms** course, Big O is the "measuring stick" for these chefs. It doesn't tell us exactly how many seconds a program takes to run (because a faster computer is just a faster chef), but it tells us **how the workload grows** as the input size () increases.

1. **Predicting the Future:** As a **Data Science** student, you'll work with massive datasets. An algorithm that works for 100 rows of data might "crash" your computer if you try to run it on 1,000,000 rows if its complexity is .


2. **Hardware vs. Efficiency:** You can buy a faster MacBook Pro, but a fast computer running a slow  algorithm will eventually be beaten by a slow computer running a fast  algorithm as  gets large.
3. **Choosing the Right Tool:** In your lab, you are implementing **Counting Sort** and **Radix Sort**. These are special because they can achieve **Linear Time ()** under the right conditions, which is significantly faster than the  of **Insertion Sort** for large lists.



Understanding Big O is the difference between a program that finishes in a heartbeat and one that keeps the user waiting forever.

Since your assignment is due soon, would you like me to help you double-check the Big O analysis for the **Counting Sort** and **Radix Sort** methods you just finished writing?
"""

# A user question we want the system to answer.
user_query = "What is the role of heaps?"

print("📄 Document preview (first 500 characters):\n")
print(doc_text[:500], "...\n")

print("❓ User query:")
print(user_query)

📄 Document preview (first 500 characters):


Imagine you are the head chef at a massive gala in **Boston**, and your job is to prepare appetizers for the guests.

### The Tale of Two Chefs

**Chef Constant ()**
Chef Constant has a signature move: no matter how many guests arrive, he always puts a single cherry on top of a pre-made cake. Whether there is 1 guest or 1,000, it takes him the exact same amount of time. He is remarkably efficient because his work doesn't scale with the size of the crowd.

**Chef Linear ()**
Chef Linear is assig ...

❓ User query:
What is the role of heaps?


In [ ]:
# Cell 3: Chunk the document into smaller pieces

from textwrap import wrap

# Simple manual chunking just for teaching:
# We'll break the document into chunks of about N characters.
# In real systems, we often chunk by tokens or sentences.
CHUNK_SIZE = 800
CHUNK_OVERLAP = 50

def simple_char_chunk(text, chunk_size=400, overlap=50):
    """
    Very simple character-based chunking.
    Not production-ready, but good for teaching.
    """
    chunks = []
    start = 0
    text_length = len(text)
    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start = end - overlap  # step with overlap
    return chunks

chunks = simple_char_chunk(doc_text, CHUNK_SIZE, CHUNK_OVERLAP)

print(f"🔹 Number of chunks created: {len(chunks)}\n")
for i, ch in enumerate(chunks, start=1):
    print(f"--- Chunk {i} ---")
    print(ch[:250], "...\n")  # show only first 250 chars for brevity

🔹 Number of chunks created: 4

--- Chunk 1 ---
Imagine you are the head chef at a massive gala in **Boston**, and your job is to prepare appetizers for the guests.

### The Tale of Two Chefs

**Chef Constant ()**
Chef Constant has a signature move: no matter how many guests arrive, he always puts ...

--- Chunk 2 ---
*Chef Quadratic ()**
Chef Quadratic is in charge of "Social Networking." He has to introduce every guest to every other guest.

* If there are 2 guests, he makes 1 introduction.
* If there are 10 guests, he makes nearly 100.
* If 1,000 guests show up ...

--- Chunk 3 ---
As a **Data Science** student, you'll work with massive datasets. An algorithm that works for 100 rows of data might "crash" your computer if you try to run it on 1,000,000 rows if its complexity is .


2. **Hardware vs. Efficiency:** You can buy a f ...

--- Chunk 4 ---
n a heartbeat and one that keeps the user waiting forever.

Since your assignment is due soon, would you like me to help you double-check

In [ ]:
# Cell 4: Configure LlamaIndex to use Ollama locally (no OpenAI)

from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

# Make sure the Ollama server is running in the background (ollama serve).
# Also ensure you have pulled the necessary models:
#   ollama pull llama3
#   ollama pull nomic-embed-text

# Set the LLM to use Ollama's "llama3" model
Settings.llm = Ollama(model="llama3")

# Set the embedding model to use Ollama's "nomic-embed-text" model
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text")

print("✅ LlamaIndex is now configured to use Ollama for both LLM and embeddings (locally).")

✅ LlamaIndex is now configured to use Ollama for both LLM and embeddings (locally).


In [ ]:
# Cell 5: Build a vector index from the chunks and run a RAG-style query

from llama_index.core import Document, VectorStoreIndex

# Wrap each chunk as a LlamaIndex Document.
# Note: Document expects keyword arguments (text=...).
documents = [Document(text=chunk) for chunk in chunks]

# Build a vector index from the documents.
# Under the hood:
# - Each chunk is embedded via OllamaEmbedding.
# - The embeddings are stored in a vector index.
index = VectorStoreIndex.from_documents(documents)

# Create a query engine.
query_engine = index.as_query_engine()

# Ask our user query.
response = query_engine.query(user_query)

print("❓ Query:")
print(user_query)
print("\n🧠 RAG-style answer (LlamaIndex + Ollama):\n")
print(response)

❓ Query:
What is the role of heaps?

🧠 RAG-style answer (LlamaIndex + Ollama):

The concept of Big O, which measures how the workload grows as input size increases, is crucial in understanding the efficiency of algorithms. In this scenario, it's essential to choose the right tool for the job, considering factors like hardware and algorithm complexity.


### Exercise 1 – Change the Document

1. Go back to **Cell 2** and replace `doc_text` with your own text, for example:
   - A section from a textbook,
   - A class reading,
   - A web article (copied as plain text).
2. Keep the rest of the notebook the same.
3. Rerun **Cell 2 → 3 → 4 → 5**.

**Questions:**
- How many chunks are created now?
- Does the answer from the RAG pipeline correctly reflect the content of your new document?

### Exercise 2 – Experiment with Chunk Size

1. In **Cell 3**, change `CHUNK_SIZE` and `CHUNK_OVERLAP`, for example:
   - Small chunks: `CHUNK_SIZE = 200`, `CHUNK_OVERLAP = 20`
   - Larger chunks: `CHUNK_SIZE = 800`, `CHUNK_OVERLAP = 100`
2. Rerun **Cell 3 → 5**.

**Questions:**
- Do smaller chunks make the answer more specific or more fragmented?
- Do larger chunks risk including irrelevant parts of the text?
- How might this relate to the **context window** of the LLM?

### Exercise 3 – Ask Different Questions

1. In **Cell 2**, keep the same `doc_text` but change `user_query` to questions like:
   - "What is an embedding and why is it useful?"
   - "How does RAG reduce hallucinations in language models?"
   - "What is the role of chunking in this system?"
2. Rerun **Cell 2 → 5**.

**Questions:**
- Does the model’s answer stay grounded in the text?
- Can you find a question where the model starts to guess beyond the given text?